# CW network - training on real recordings, run 2

**Before you start:** upload `cw_code.zip` and the new `cw_init.pt` (replace the old one; `cw_real.zip` is already there) to the main folder of your Google Drive (My Drive).

1. Menu **Runtime > Change runtime type** - choose **T4 GPU** - Save.
2. Run the boxes one by one (the play button on each). The first one asks for access to your Drive - allow it.
3. Box 4 is the training, under an hour. If Colab stops it, run boxes 1, 2, 3 and 4 again - it goes on from where it stopped.
4. The result is in your Drive, folder **cw_real2_out**. When it has finished, tell Claude.

In [ ]:
# 1. Connect Google Drive and unpack the files
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil
os.makedirs('/content/cw', exist_ok=True)
zipfile.ZipFile('/content/drive/MyDrive/cw_real.zip').extractall('/content/cw')
if os.path.exists('/content/drive/MyDrive/cw_code.zip'):   # newer code, so the big zip need not go up again
    zipfile.ZipFile('/content/drive/MyDrive/cw_code.zip').extractall('/content/cw')
shutil.copy('/content/drive/MyDrive/cw_init.pt', '/content/cw/init.pt')
print(os.listdir('/content/cw/deepnet'), os.listdir('/content/cw/data'))

In [ ]:
# 2. Is the GPU there?
import torch, os
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('CPU cores:', os.cpu_count())

In [ ]:
# 3. A short test (a few minutes): it should print steps and HELD-OUT lines
!python /content/cw/deepnet/finetune_gpu.py --own /content/cw/data/own_cut --w1aw /content/cw/data/w1aw_cut --init /content/cw/init.pt --out /content/test_out --steps 50 --workers 2 2>&1 | grep -v -i warn | tail -6

In [ ]:
# 4. THE TRAINING. Saves every 250 steps into your Drive (folder cw_real2_out).
#    If it stops, run boxes 1-3 again and then this box again - it goes on from the last save.
!python /content/cw/deepnet/finetune_gpu.py --own /content/cw/data/own_cut --w1aw /content/cw/data/w1aw_cut --init /content/cw/init.pt --out /content/drive/MyDrive/cw_real2_out --steps 3000 --workers 2 2>&1 | grep -v -i warn

In [ ]:
# 5. Look at how it is doing (held-out error every 250 steps - lower is better)
!grep HELD /content/drive/MyDrive/cw_real2_out/train.log | tail -12